In [1]:
## DELTAKIT STIM ##
import deltakit_stim
# Deltakit circuit clashes with deltakit_stim unless do these injections
injections = ['stim._detect_machine_architecture', 'stim._stim_polyfill', 'stim']
import sys
for namespace in injections:
    sys.modules[namespace] = sys.modules[f"deltakit_{namespace}"]


## BB_IONS ##
import os
sys.path.append(os.path.abspath("../src"))
from bb_ions import *

In [6]:
# Trying this on one of my stim circuits:

code = bb6_18_4_4_code()
p = 0.04
memory_basis = 'X'

circuit = make_BB_circuit(  # see src/bb_ions/circfuncs for explanation of make_BB_circuit inputs
    code,
    p,  
    errors = helios_errors(p),
    idle_during = helios_idle_errors(p),
    num_syndrome_extraction_cycles = 10,  
    memory_basis = memory_basis,
    sequential_gates = False,
    exclude_opposite_basis_detectors = False,
    reuse_check_qubits = True,
    swap_LRC = True,
    only_CZs = True,
    leakage = True
)

# svg = circuit.diagram('timeline-svg')
# display(svg)
# with open(f"../scrap.svg", "w", encoding="utf-8") as f: f.write(str(svg))

### Decoding

In [3]:
## Rebecca's settings:
                # max_bp_iters = 10, # default 30
                # bp_method="minimum_sum", # product_sum (default), min_sum, min_sum_log
                # ms_scaling_factor = 0.625, # normalisation
                # schedule="serial", 
                # osd_method="osd_cs", # "osd0" - zero-order OSD, "osd_e" - exhaustive OSD, "osd_cs": combination-sweep OSD (default)
                # osd_order=9

In [7]:
# Example based on example in README on https://github.com/oscarhiggott/stimbposd

from stimbposd import BPOSD

num_shots = 5

detector_sampler = circuit.compile_detector_sampler()
detection_events, observables = detector_sampler.sample(shots = num_shots, separate_observables=True)

decoder = BPOSD(
    circuit.detector_error_model(approximate_disjoint_errors = True),  ## THIS NEEDED TO BE TRUE BECAUSE WE'RE USING PAULI_CHANNEL_2 -- check what this approximation does.
    max_bp_iters=1000, 
    bp_method = "min_sum",
    osd_method = "osd_cs",
    osd_order = 5
    
    # # Rebecca's settings:
    # max_bp_iters = 10, # default 30
    # bp_method="minimum_sum", # product_sum (default), min_sum, min_sum_log
    # ms_scaling_factor = 0.625, # normalisation
    # schedule="serial", 
    # osd_method="osd_cs", # "osd0" - zero-order OSD, "osd_e" - exhaustive OSD, "osd_cs": combination-sweep OSD (default)
    # osd_order=9
    )

predicted_observables = decoder.decode_batch(detection_events)
print(predicted_observables)
print(observables)

num_mistakes = np.sum(np.any(predicted_observables != observables, axis=1))

print(f"num_mistakes = {num_mistakes}/{num_shots}")

[[ True  True  True False]
 [False  True  True  True]
 [False  True False False]
 [False False  True  True]
 [False  True False False]]
[[ True  True  True  True]
 [ True  True False  True]
 [False  True False  True]
 [ True  True False  True]
 [ True  True False  True]]
num_mistakes = 5/5


In [5]:
circuit = stim.Circuit('''
R 0 1
HERALDED
LEAKAGE(0.05) 1
CX 0 1
M 0 1
HERALD_LEAKAGE_EVENT() 0 1
DETECTOR rec[-4]
DETECTOR rec[-3]
DETECTOR rec[-2]
DETECTOR rec[-1]
''')

ValueError: Gate not found: 'HERALDED'

### Working with Sinter

In [ ]:
# Need to run ../scripts/run_with_leakage.py (deltakit_stim clashes with imports within sinter otherwise)

### Heralded_erase for loss

Need to feed the decoder probability priors to signify a heralded erase has occurred (just passing the DEM won't do it)

In [ ]:
circuit = stim.Circuit("""
    HERALDED_ERASE(0.5) 0
    DETECTOR rec[-1]  # Heralod 0
    HERALDED_ERASE(0.5) 0
    DETECTOR rec[-1]   # Heralod 1
    HERALDED_ERASE(0.5) 0
    DETECTOR rec[-1]  # Heralod 2
""")

detector_sampler = circuit.compile_detector_sampler()

num_shots = 1

detection_events, observables = detector_sampler.sample(shots = num_shots, separate_observables=True)

print(detection_events)


[[False  True  True]]


In [ ]:
import deltakit_stim
import sys
sys.modules["stim"] = deltakit_stim

import sinter
import numpy as np
import glob
from stimbposd import SinterDecoder_BPOSD, sinter_decoders
import time
import os
import multiprocessing

sys.path.append(os.path.abspath("../src"))
from bb_ions import *


In [ ]:
code = bb5_12_4_2()
circuit = make_BB_circuit(code)
dem = circuit.detector_error_model(approximate_disjoint_errors = True)
# print(dem)

dem_matrices = stimbposd.detector_error_model_to_check_matrices(dem)
H = dem_matrices.check_matrix.toarray() # note this is NOT the code parity-check matrix. It is the dem matrix. Columns are circuit error mechanisms, rows are detectors

# Create a decoder every shot? I read somewhere someone saying there might be a way to avoid this

NameError: name 'stimbposd' is not defined

https://github.com/oscarhiggott/stimbposd/blob/5ed95e415be245f93d5326c4898bd0182fe8f3bd/src/stimbposd/bp_osd.py#L13

In [ ]:
# Chatgippity BS:

import stim
import numpy as np

from ldpc.bposd_decoder import BpOsdDecoder
from stimbposd.dem_to_matrices import detector_error_model_to_check_matrices


# ============================================
# 1. Build a tiny circuit with heralded erasures
# ============================================

circuit = stim.Circuit("""
R 0 1
HERALDED_ERASE(0.2) 0
HERALDED_ERASE(0.2) 1
M 0 1
DETECTOR rec[-4]
DETECTOR rec[-3]
DETECTOR rec[-2]
DETECTOR rec[-1]
""")


# ============================================
# 2. Build DEM and parity-check matrix
# ============================================

dem = circuit.detector_error_model(approximate_disjoint_errors = True)

matrices = 

In [ ]:

matrices = detector_error_model_to_check_matrices(
    dem,
    allow_undecomposed_hyperedges=True,
)

H = matrices.check_matrix


# ============================================
# 3. Create ONE decoder object
# ============================================

base_p = 0.001

decoder = BpOsdDecoder(
    pcm=H,
    channel_probs=np.full(H.shape[1], base_p),
    max_iter=20,
    bp_method="product_sum",
    osd_method="osd_cs",
    osd_order=10,
)


# ============================================
# 4. Sample ACTUAL CIRCUIT shots
# ============================================

sampler = circuit.compile_detector_sampler()

shots = sampler.sample(
    shots=5,
)

print("Detector shots:")
print(shots.astype(int))


# ============================================
# 5. Decode each shot using herald info
# ============================================

for shot_index, syndrome in enumerate(shots):

    # Start from default priors
    priors = np.full(H.shape[1], base_p)

    # ----------------------------------------
    # Herald detectors:
    #
    # detector 0 -> erasure on qubit 0
    # detector 1 -> erasure on qubit 1
    # ----------------------------------------

    erased_qubits = []

    if syndrome[0]:
        erased_qubits.append(0)

    if syndrome[1]:
        erased_qubits.append(1)

    print(f"\nShot {shot_index}")
    print("Erased qubits:", erased_qubits)

    # ========================================
    # IMPORTANT PART:
    #
    # Convert heralds into BP priors
    # ========================================

    #
    # In a real circuit-level decoder,
    # you would map erased qubits onto
    # the corresponding DEM columns.
    #
    # Here we do a toy example:
    #

    for q in erased_qubits:

        #
        # Find DEM columns associated
        # with this erased qubit.
        #
        # For this tiny toy example,
        # assume:
        #
        # qubit 0 -> column 0
        # qubit 1 -> column 1
        #

        priors[q] = 0.5

    # Reuse existing decoder object
    decoder.channel_probs = priors

    # Decode
    correction = decoder.decode(syndrome)

    print("Syndrome:", syndrome.astype(int))
    print("Correction:", correction)